# 🚀 SLM Fine-Tuning for RAG — Google Colab Training

**Fine-tuning Qwen3-1.7B with QLoRA for RAG on Italian Public Administration Documents**

This notebook runs the training phase on a free T4 GPU.

## Prerequisites
1. Upload your project to Google Drive OR clone from GitHub
2. Ensure you have already run `preprocess` and `build-index` locally
3. Runtime → Change runtime type → T4 GPU

## 1. Setup Environment

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# Set project path — UPDATE THIS to your project location in Google Drive
import os

PROJECT_PATH = '/content/drive/MyDrive/SLM_Finetuning_4_RAG'  # <-- Update this!

# Alternatively, clone from GitHub:
# !git clone https://github.com/YOUR_USERNAME/SLM_Finetuning_4_RAG.git
# PROJECT_PATH = '/content/SLM_Finetuning_4_RAG'

os.chdir(PROJECT_PATH)
print(f'Working directory: {os.getcwd()}')
!ls -la

In [ ]:
# Install dependencies
!pip install -q torch transformers accelerate datasets peft trl bitsandbytes \
    sentence-transformers faiss-cpu PyMuPDF ragas langchain langchain-community \
    langchain-huggingface matplotlib seaborn pandas scikit-learn tqdm

In [ ]:
# Verify GPU
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 2. Data Preprocessing (if not done locally)

Skip this section if you already ran `python -m src.main preprocess` and `python -m src.main build-index` locally and uploaded the `outputs/` folder.

In [ ]:
# Optional: Run preprocessing if not already done
import sys
sys.path.insert(0, PROJECT_PATH)

# Check if preprocessing was already done
from pathlib import Path
if not (Path('outputs/qa_dataset/sft_dataset').exists()):
    print('Running preprocessing pipeline...')
    from src.data.pdf_extractor import extract_all_pdfs
    from src.data.chunker import chunk_documents
    from src.data.qa_generator import generate_qa_dataset
    from src.data.dataset_builder import build_dataset
    from src.config import get_config
    
    config = get_config()
    docs = extract_all_pdfs(config)
    chunks = chunk_documents(config)
    qa_pairs = generate_qa_dataset(config)
    dataset = build_dataset(config)
    print(f'Done! {len(qa_pairs)} QA pairs, {len(dataset["train"])} training samples')
else:
    print('Preprocessing already done! Skipping...')

## 3. QLoRA Fine-Tuning

In [ ]:
import sys
sys.path.insert(0, PROJECT_PATH)

import logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)s | %(message)s')

from src.config import get_config
from src.training.train import train_model

config = get_config()

# On T4 (16GB VRAM) we can use slightly larger batch size
config.training.per_device_train_batch_size = 2
config.training.gradient_accumulation_steps = 4
config.training.num_train_epochs = 3

print(f'Training config:')
print(f'  Batch size: {config.training.per_device_train_batch_size}')
print(f'  Gradient accumulation: {config.training.gradient_accumulation_steps}')
print(f'  Effective batch: {config.training.per_device_train_batch_size * config.training.gradient_accumulation_steps}')
print(f'  Epochs: {config.training.num_train_epochs}')
print(f'  Learning rate: {config.training.learning_rate}')
print(f'  LoRA r: {config.lora.r}, alpha: {config.lora.lora_alpha}')

In [ ]:
# Run training
metrics = train_model(config)

print('\n' + '='*60)
print('Training Complete!')
print('='*60)
for k, v in metrics.items():
    print(f'  {k}: {v}')

## 4. Build Vector Index (if not done)

In [ ]:
from pathlib import Path
from src.config import get_config

config = get_config()
if not (config.paths.vectorstore_dir / 'index.faiss').exists():
    print('Building FAISS index...')
    from src.rag.vectorstore import build_vectorstore
    vs = build_vectorstore(config)
    print('Done!')
else:
    print('FAISS index already exists!')

## 5. Evaluation

In [ ]:
from src.evaluation.evaluate import evaluate_rag
from src.config import get_config
import json

config = get_config()

# Evaluate base model
print('Evaluating BASE model...')
base_scores = evaluate_rag(config, use_finetuned=False)
print(f'Base scores: {json.dumps(base_scores["aggregate_scores"], indent=2)}')

print('\n' + '='*60)

# Evaluate fine-tuned model
print('Evaluating FINE-TUNED model...')
ft_scores = evaluate_rag(config, use_finetuned=True)
print(f'Fine-tuned scores: {json.dumps(ft_scores["aggregate_scores"], indent=2)}')

## 6. Comparison & Plots

In [ ]:
from src.evaluation.compare import run_comparison
from src.config import get_config

config = get_config()
summary = run_comparison(config)

# Display plots inline
from IPython.display import Image, display
from pathlib import Path

plots_dir = config.paths.plots_dir
for plot_file in sorted(plots_dir.glob('*.png')):
    print(f'\n--- {plot_file.name} ---')
    display(Image(filename=str(plot_file), width=700))

## 7. Interactive Demo

In [ ]:
from src.rag.pipeline import RAGPipeline
from src.config import get_config

config = get_config()

# Load fine-tuned pipeline
pipeline = RAGPipeline(config=config, use_finetuned=True)

# Test questions
test_questions = [
    'Quali sono le procedure previste per gli appalti pubblici?',
    'Cosa prevede la normativa sulle firme elettroniche?',
    'Quali sono gli obblighi delle stazioni appaltanti?',
    'Come funziona il sistema di qualificazione delle imprese?',
]

for q in test_questions:
    result = pipeline.query(q)
    print(f'\n{"="*60}')
    print(f'Q: {result["question"]}')
    print(f'A: {result["answer"][:300]}...')
    print(f'Sources: {result["source_documents"]}')
    print(f'Time: {result["total_time_s"]}s')

pipeline.cleanup()

## 8. Download Results

If you trained on Colab, download the model and results back to your local machine.

In [ ]:
# Zip outputs for download
import shutil
shutil.make_archive('/content/slm_rag_outputs', 'zip', 'outputs')

from google.colab import files
files.download('/content/slm_rag_outputs.zip')
print('Download started!')